In [ ]:
# --- repo bootstrap ---
import sys
from pathlib import Path

repo = Path.cwd()
if repo.name == "notebooks":
    repo = repo.parent

if not (repo / "src").exists():
    !git clone https://github.com/thinkthoughts/ion-transport-waveform-pipeline.git
    %cd ion-transport-waveform-pipeline
    repo = Path.cwd()

if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

print("Repo root:", repo)


# 03 — Waveform Generation

Map a smooth transport path x_c(t) to electrode voltage waveforms v(t).

```text
transport path → voltage solver → v(t) → continuity + constraints
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.ion_transport_waveform.config import TrapConfig
from src.ion_transport_waveform.trap_model import gaussian_electrode_basis
from src.ion_transport_waveform.transport_path import minimum_jerk_path
from src.ion_transport_waveform.waveform_solver import waveform_for_path

cfg = TrapConfig()
fig_dir = repo / "figures"
fig_dir.mkdir(exist_ok=True)


## 1. Build basis and path


In [ ]:
x = np.linspace(-420e-6, 420e-6, 800)
electrode_positions = np.arange(-5,6) * cfg.electrode_pitch_m
basis = gaussian_electrode_basis(x, electrode_positions, cfg.basis_width_m)

t = np.linspace(0, 20e-6, 400)
path = minimum_jerk_path(t, -160e-6, 160e-6, duration=20e-6)


## 2. Generate waveform


In [ ]:
V = waveform_for_path(basis, x, path, voltage_limit=cfg.voltage_limit_v)
print("waveform shape:", V.shape)


## 3. Plot voltage waveforms


In [ ]:
plt.figure(figsize=(8,4.5))
for i in range(V.shape[1]):
    plt.plot(t*1e6, V[:,i], alpha=0.6)
plt.xlabel("time (µs)")
plt.ylabel("voltage (V)")
plt.title("Electrode voltage waveforms")
plt.tight_layout()
plt.savefig(fig_dir / "03_waveform_profiles.png", dpi=180)
plt.show()


## 4. Continuity check (ΔV)


In [ ]:
dV = np.diff(V, axis=0)
max_step = np.max(np.abs(dV))
print("max ΔV between steps:", max_step)

plt.figure(figsize=(8,4))
plt.plot(np.max(np.abs(dV), axis=1))
plt.xlabel("time index")
plt.ylabel("max |ΔV|")
plt.title("Waveform step size (continuity check)")
plt.tight_layout()
plt.savefig(fig_dir / "03_waveform_step_size.png", dpi=180)
plt.show()


## 5. Save waveform


In [ ]:
np.savez(repo / "data" / "simulation_outputs" / "waveform_03.npz",
         t=t, path=path, voltages=V)
print("saved waveform data")


## Next

Notebook 04: simulate ion motion using these waveforms.
